###1) Parametry i zakres runów pogodowych

In [0]:
from datetime import (
    datetime,
    date,
    time,
    timezone,
    timedelta,
)
from pathlib import Path
from zoneinfo import ZoneInfo

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

# Te parametry będzie później przekazywał Lakeflow Job.
dbutils.widgets.text(
    "evaluation_start_date_local",
    "2024-07-01"
)

dbutils.widgets.text(
    "evaluation_end_date_local_exclusive",
    "2024-11-01"
)

EVALUATION_START_DATE_TEXT = dbutils.widgets.get(
    "evaluation_start_date_local"
)

EVALUATION_END_DATE_TEXT = dbutils.widgets.get(
    "evaluation_end_date_local_exclusive"
)

MARKET_TIMEZONE = "America/Los_Angeles"
LOCAL_TIMEZONE = ZoneInfo(MARKET_TIMEZONE)

MODEL = "ecmwf_ifs"
FORECAST_HOURS = 72
PUBLICATION_LAG_HOURS = 6

LATITUDE = 38.996306
LONGITUDE = -122.134111

RUNS_DIRECTORY = Path(
    "/Volumes/workspace/energy_bronze/landing/"
    "open_meteo/ecmwf_ifs_72h_runs"
)

RUNS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

evaluation_start_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_START_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

evaluation_end_local = datetime.combine(
    date.fromisoformat(
        EVALUATION_END_DATE_TEXT
    ),
    time.min,
    tzinfo=LOCAL_TIMEZONE
)

if evaluation_end_local <= evaluation_start_local:
    raise ValueError(
        "Data końcowa musi być późniejsza "
        "od daty początkowej."
    )

EVALUATION_START_UTC = (
    evaluation_start_local
    .astimezone(timezone.utc)
)

EVALUATION_END_UTC = (
    evaluation_end_local
    .astimezone(timezone.utc)
)

# Pierwsze zamknięcie ofert FMM przypada
# 75 minut przed pierwszą godziną dostawy.
first_decision_utc = (
    EVALUATION_START_UTC
    - timedelta(minutes=75)
)

last_target_interval_start_utc = (
    EVALUATION_END_UTC
    - timedelta(minutes=15)
)

last_operating_hour_utc = (
    last_target_interval_start_utc
    .replace(
        minute=0,
        second=0,
        microsecond=0
    )
)

last_decision_utc = (
    last_operating_hour_utc
    - timedelta(minutes=75)
)


def floor_to_ecmwf_run(timestamp_utc):
    timestamp_utc = timestamp_utc.astimezone(
        timezone.utc
    )

    run_hour = (
        12
        if timestamp_utc.hour >= 12
        else 0
    )

    return timestamp_utc.replace(
        hour=run_hour,
        minute=0,
        second=0,
        microsecond=0
    )


# Run musi być opublikowany przed decyzją.
WEATHER_RUN_START_UTC = (
    floor_to_ecmwf_run(
        first_decision_utc
        - timedelta(hours=PUBLICATION_LAG_HOURS)
    )
    - timedelta(hours=12)
)

WEATHER_RUN_END_UTC = floor_to_ecmwf_run(
    last_decision_utc
    - timedelta(hours=PUBLICATION_LAG_HOURS)
)

EXPECTED_RUN_TIMES = []

current_run_utc = WEATHER_RUN_START_UTC

while current_run_utc <= WEATHER_RUN_END_UTC:
    EXPECTED_RUN_TIMES.append(
        current_run_utc
    )

    # ECMWF IFS w wykorzystanym archiwum
    # udostępnia runy 00 i 12 UTC.
    current_run_utc += timedelta(hours=12)

EXPECTED_RUN_COUNT = len(EXPECTED_RUN_TIMES)

EXPECTED_FORECAST_ROW_COUNT = (
    EXPECTED_RUN_COUNT
    * FORECAST_HOURS
)

print(
    "Zakres lokalny:",
    evaluation_start_local,
    "—",
    evaluation_end_local
)

print(
    "Zakres UTC:",
    EVALUATION_START_UTC,
    "—",
    EVALUATION_END_UTC
)

print(
    "Zakres runów pogodowych:",
    WEATHER_RUN_START_UTC,
    "—",
    WEATHER_RUN_END_UTC
)

print(
    "Oczekiwana liczba runów:",
    EXPECTED_RUN_COUNT
)

print(
    "Oczekiwana liczba rekordów Bronze:",
    EXPECTED_FORECAST_ROW_COUNT
)

print("Katalog:", RUNS_DIRECTORY)

###2) Wznawialne pobieranie runów pogodowych

In [0]:
import json
import time as time_module
from collections import Counter

import requests

ENDPOINT = (
    "https://single-runs-api.open-meteo.com/"
    "v1/forecast"
)

HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
    "shortwave_radiation_instant",
    "global_tilted_irradiance_instant",
    "is_day",
]

http_session = requests.Session()

http_session.headers.update({
    "User-Agent": (
        "master-thesis-energy-pipeline/2.0"
    )
})


def target_file_for_run(run_time_utc):
    return (
        RUNS_DIRECTORY
        / (
            f"{MODEL}_"
            f"{run_time_utc.strftime('%Y%m%dT%H%MZ')}_"
            f"{FORECAST_HOURS}h.json"
        )
    )


def valid_weather_artifact(
    file_path,
    expected_run_time_utc
):
    if not file_path.exists():
        return False

    if file_path.stat().st_size == 0:
        return False

    try:
        with file_path.open(
            "r",
            encoding="utf-8"
        ) as artifact_file:
            artifact = json.load(artifact_file)

        payload = artifact["response"]
        request_parameters = artifact[
            "request_parameters"
        ]

        received_times = payload[
            "hourly"
        ]["time"]

        artifact_run_text = request_parameters[
            "run"
        ]

        expected_run_text = (
            expected_run_time_utc.strftime(
                "%Y-%m-%dT%H:%M"
            )
        )

        return (
            len(received_times) == FORECAST_HOURS
            and artifact_run_text
                == expected_run_text
        )

    except Exception:
        return False


download_results = []

for run_number, run_time_utc in enumerate(
    EXPECTED_RUN_TIMES,
    start=1
):
    target_file = target_file_for_run(
        run_time_utc
    )

    if valid_weather_artifact(
        target_file,
        run_time_utc
    ):
        status = "skipped_existing"
        error_text = ""

    else:
        temporary_file = target_file.with_suffix(
            target_file.suffix + ".part"
        )

        temporary_file.unlink(
            missing_ok=True
        )

        run_text = run_time_utc.strftime(
            "%Y-%m-%dT%H:%M"
        )

        request_parameters = {
            "latitude": LATITUDE,
            "longitude": LONGITUDE,
            "models": MODEL,
            "run": run_text,
            "forecast_hours": FORECAST_HOURS,
            "hourly": ",".join(
                HOURLY_VARIABLES
            ),
            "timezone": "GMT",
            "timeformat": "iso8601",
            "wind_speed_unit": "ms",
            "tilt": 25,
            "azimuth": 0,
        }

        status = "failed"
        error_text = ""

        for attempt in range(1, 5):
            try:
                response = http_session.get(
                    ENDPOINT,
                    params=request_parameters,
                    timeout=(10, 180)
                )

                response.raise_for_status()
                payload = response.json()

                received_times = (
                    payload
                    .get("hourly", {})
                    .get("time", [])
                )

                if (
                    len(received_times)
                    != FORECAST_HOURS
                ):
                    raise RuntimeError(
                        "Otrzymano "
                        f"{len(received_times)} godzin "
                        "zamiast "
                        f"{FORECAST_HOURS}."
                    )

                artifact = {
                    "source": (
                        "Open-Meteo "
                        "Single Runs API"
                    ),
                    "request_parameters":
                        request_parameters,
                    "requested_url":
                        response.url,
                    "retrieved_at_utc":
                        datetime.now(
                            timezone.utc
                        ).isoformat(),
                    "response": payload,
                }

                with temporary_file.open(
                    "w",
                    encoding="utf-8"
                ) as artifact_file:
                    json.dump(
                        artifact,
                        artifact_file,
                        ensure_ascii=False,
                        indent=2
                    )

                if not valid_weather_artifact(
                    temporary_file,
                    run_time_utc
                ):
                    raise RuntimeError(
                        "Zapisany plik nie przeszedł "
                        "kontroli jakości."
                    )

                temporary_file.replace(
                    target_file
                )

                status = "downloaded"
                error_text = ""
                break

            except Exception as error:
                error_text = (
                    f"{type(error).__name__}: "
                    f"{str(error)}"
                )

                temporary_file.unlink(
                    missing_ok=True
                )

                if attempt < 4:
                    wait_seconds = 2 ** attempt

                    print(
                        f"{run_number:02d}/"
                        f"{EXPECTED_RUN_COUNT} "
                        f"{run_time_utc.isoformat()} "
                        f"próba {attempt}/4 "
                        f"nieudana: {error_text}. "
                        f"Ponowienie za "
                        f"{wait_seconds} s."
                    )

                    time_module.sleep(
                        wait_seconds
                    )

        time_module.sleep(0.35)

    download_results.append({
        "run_time_utc": run_time_utc,
        "target_file": str(target_file),
        "status": status,
        "error": error_text,
    })

    print(
        f"{run_number:02d}/"
        f"{EXPECTED_RUN_COUNT} "
        f"{run_time_utc.isoformat()} "
        f"{status}"
    )


status_counts = Counter(
    result["status"]
    for result in download_results
)

valid_target_files = [
    target_file_for_run(run_time_utc)
    for run_time_utc in EXPECTED_RUN_TIMES
    if valid_weather_artifact(
        target_file_for_run(run_time_utc),
        run_time_utc
    )
]

failed_results = [
    result
    for result in download_results
    if result["status"] == "failed"
]

print()
print("=" * 75)
print("PODSUMOWANIE POBIERANIA POGODY")
print("=" * 75)

print(
    "Pominięte istniejące:",
    status_counts.get(
        "skipped_existing",
        0
    )
)

print(
    "Pobrane:",
    status_counts.get(
        "downloaded",
        0
    )
)

print(
    "Nieudane:",
    status_counts.get(
        "failed",
        0
    )
)

print(
    "Poprawne pliki:",
    f"{len(valid_target_files)}/"
    f"{EXPECTED_RUN_COUNT}"
)

if failed_results:
    print("\nNieudane runy:")

    for result in failed_results:
        print(
            result["run_time_utc"],
            result["error"]
        )

if (
    len(valid_target_files)
    != EXPECTED_RUN_COUNT
):
    raise RuntimeError(
        "Pobieranie niekompletne. "
        "Uruchom tę komórkę ponownie; "
        "poprawne pliki zostaną pominięte."
    )

print(
    "Pobieranie zakończone: "
    "wszystkie runy są dostępne."
)

###3) Spłaszczenie JSON

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType,
    BooleanType,
    TimestampType,
)


def parse_utc(timestamp_text):
    parsed_timestamp = datetime.fromisoformat(
        str(timestamp_text).replace(
            "Z",
            "+00:00"
        )
    )

    if parsed_timestamp.tzinfo is None:
        parsed_timestamp = (
            parsed_timestamp.replace(
                tzinfo=timezone.utc
            )
        )

    return parsed_timestamp.astimezone(
        timezone.utc
    )


def float_value_at(
    hourly_payload,
    variable_name,
    index
):
    values = hourly_payload.get(
        variable_name
    )

    if values is None:
        return None

    value = values[index]

    if value is None:
        return None

    return float(value)


def integer_value_at(
    hourly_payload,
    variable_name,
    index
):
    values = hourly_payload.get(
        variable_name
    )

    if values is None:
        return None

    value = values[index]

    if value is None:
        return None

    return int(value)


input_files = [
    target_file_for_run(run_time_utc)
    for run_time_utc in EXPECTED_RUN_TIMES
]

invalid_input_files = [
    file_path
    for file_path, run_time_utc in zip(
        input_files,
        EXPECTED_RUN_TIMES
    )
    if not valid_weather_artifact(
        file_path,
        run_time_utc
    )
]

if invalid_input_files:
    raise RuntimeError(
        "Nie wszystkie pliki wejściowe "
        "są poprawne: "
        + ", ".join(
            file_path.name
            for file_path
            in invalid_input_files[:10]
        )
    )

forecast_rows = []

for raw_file, expected_run_time_utc in zip(
    input_files,
    EXPECTED_RUN_TIMES
):
    with raw_file.open(
        "r",
        encoding="utf-8"
    ) as artifact_file:
        artifact = json.load(
            artifact_file
        )

    payload = artifact["response"]
    request_parameters = artifact[
        "request_parameters"
    ]

    run_time_utc = parse_utc(
        request_parameters["run"]
    )

    if run_time_utc != expected_run_time_utc:
        raise RuntimeError(
            f"Niezgodny run w pliku "
            f"{raw_file.name}."
        )

    available_at_utc = (
        run_time_utc
        + timedelta(
            hours=PUBLICATION_LAG_HOURS
        )
    )

    ingested_at_utc = parse_utc(
        artifact["retrieved_at_utc"]
    )

    hourly_payload = payload["hourly"]
    hourly_times = hourly_payload["time"]

    if len(hourly_times) != FORECAST_HOURS:
        raise RuntimeError(
            f"Plik {raw_file.name} zawiera "
            f"{len(hourly_times)} godzin "
            f"zamiast {FORECAST_HOURS}."
        )

    for index, timestamp_text in enumerate(
        hourly_times
    ):
        valid_time_utc = parse_utc(
            timestamp_text
        )

        forecast_rows.append({
            "source_system": "OPEN_METEO",
            "forecast_model": MODEL,
            "forecast_run_id": (
                f"{MODEL}_"
                f"{run_time_utc.strftime('%Y%m%dT%H%MZ')}"
            ),
            "latitude": float(
                payload["latitude"]
            ),
            "longitude": float(
                payload["longitude"]
            ),
            "elevation_m": (
                None
                if payload.get("elevation")
                is None
                else float(
                    payload["elevation"]
                )
            ),
            "run_time_utc": run_time_utc,
            "available_at_utc":
                available_at_utc,
            "forecast_valid_time_utc":
                valid_time_utc,
            "lead_time_from_run_hours": (
                valid_time_utc
                - run_time_utc
            ).total_seconds() / 3600.0,
            "effective_lead_time_hours": (
                valid_time_utc
                - available_at_utc
            ).total_seconds() / 3600.0,
            "is_usable_for_replay": (
                valid_time_utc
                >= available_at_utc
            ),
            "temperature_2m_c":
                float_value_at(
                    hourly_payload,
                    "temperature_2m",
                    index
                ),
            "relative_humidity_2m_pct":
                float_value_at(
                    hourly_payload,
                    "relative_humidity_2m",
                    index
                ),
            "cloud_cover_pct":
                float_value_at(
                    hourly_payload,
                    "cloud_cover",
                    index
                ),
            "wind_speed_10m_ms":
                float_value_at(
                    hourly_payload,
                    "wind_speed_10m",
                    index
                ),
            "wind_direction_10m_deg":
                float_value_at(
                    hourly_payload,
                    "wind_direction_10m",
                    index
                ),
            "shortwave_radiation_instant_wm2":
                float_value_at(
                    hourly_payload,
                    "shortwave_radiation_instant",
                    index
                ),
            "global_tilted_irradiance_instant_wm2":
                float_value_at(
                    hourly_payload,
                    (
                        "global_tilted_"
                        "irradiance_instant"
                    ),
                    index
                ),
            "is_day":
                integer_value_at(
                    hourly_payload,
                    "is_day",
                    index
                ),
            "_source_file_path":
                str(raw_file),
            "_source_file_name":
                raw_file.name,
            "_source_file_size":
                int(
                    raw_file.stat().st_size
                ),
            "_ingested_at_utc":
                ingested_at_utc,
        })


if (
    len(forecast_rows)
    != EXPECTED_FORECAST_ROW_COUNT
):
    raise RuntimeError(
        "Niepoprawna liczba rekordów Python: "
        f"{len(forecast_rows)} zamiast "
        f"{EXPECTED_FORECAST_ROW_COUNT}."
    )


forecast_schema = StructType([
    StructField(
        "source_system",
        StringType(),
        False
    ),
    StructField(
        "forecast_model",
        StringType(),
        False
    ),
    StructField(
        "forecast_run_id",
        StringType(),
        False
    ),
    StructField(
        "latitude",
        DoubleType(),
        False
    ),
    StructField(
        "longitude",
        DoubleType(),
        False
    ),
    StructField(
        "elevation_m",
        DoubleType(),
        True
    ),
    StructField(
        "run_time_utc",
        TimestampType(),
        False
    ),
    StructField(
        "available_at_utc",
        TimestampType(),
        False
    ),
    StructField(
        "forecast_valid_time_utc",
        TimestampType(),
        False
    ),
    StructField(
        "lead_time_from_run_hours",
        DoubleType(),
        False
    ),
    StructField(
        "effective_lead_time_hours",
        DoubleType(),
        False
    ),
    StructField(
        "is_usable_for_replay",
        BooleanType(),
        False
    ),
    StructField(
        "temperature_2m_c",
        DoubleType(),
        True
    ),
    StructField(
        "relative_humidity_2m_pct",
        DoubleType(),
        True
    ),
    StructField(
        "cloud_cover_pct",
        DoubleType(),
        True
    ),
    StructField(
        "wind_speed_10m_ms",
        DoubleType(),
        True
    ),
    StructField(
        "wind_direction_10m_deg",
        DoubleType(),
        True
    ),
    StructField(
        "shortwave_radiation_instant_wm2",
        DoubleType(),
        True
    ),
    StructField(
        "global_tilted_irradiance_instant_wm2",
        DoubleType(),
        True
    ),
    StructField(
        "is_day",
        IntegerType(),
        True
    ),
    StructField(
        "_source_file_path",
        StringType(),
        False
    ),
    StructField(
        "_source_file_name",
        StringType(),
        False
    ),
    StructField(
        "_source_file_size",
        LongType(),
        False
    ),
    StructField(
        "_ingested_at_utc",
        TimestampType(),
        False
    ),
])


weather_forecast_bronze_df = (
    spark.createDataFrame(
        forecast_rows,
        schema=forecast_schema
    )
    .withColumn(
        "_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("forecast_model"),
                F.col(
                    "run_time_utc"
                ).cast("string"),
                F.col(
                    "forecast_valid_time_utc"
                ).cast("string")
            ),
            256
        )
    )
    .dropDuplicates([
        "forecast_model",
        "run_time_utc",
        "forecast_valid_time_utc"
    ])
)

bronze_dataframe_count = (
    weather_forecast_bronze_df.count()
)

print(
    "Poprawne pliki wejściowe:",
    len(input_files)
)

print(
    "Rekordy Python:",
    len(forecast_rows)
)

print(
    "Rekordy DataFrame:",
    bronze_dataframe_count
)

if (
    bronze_dataframe_count
    != EXPECTED_FORECAST_ROW_COUNT
):
    raise RuntimeError(
        "DataFrame zawiera niepoprawną "
        "liczbę rekordów."
    )

print(
    "Spłaszczenie zakończone poprawnie."
)

###4) Testy i zapis Bronze

In [0]:
BRONZE_WEATHER_TABLE = (
    "workspace.energy_bronze."
    "open_meteo_forecast_runs_raw"
)

wrong_rows_per_run_count = (
    weather_forecast_bronze_df
    .groupBy(
        "forecast_model",
        "run_time_utc"
    )
    .agg(
        F.count("*").alias("row_count")
    )
    .filter(
        F.col("row_count")
        != FORECAST_HOURS
    )
    .count()
)

duplicate_business_key_count = (
    weather_forecast_bronze_df
    .groupBy(
        "forecast_model",
        "run_time_utc",
        "forecast_valid_time_utc"
    )
    .agg(
        F.count("*").alias("row_count")
    )
    .filter(F.col("row_count") > 1)
    .count()
)

bronze_quality_df = (
    weather_forecast_bronze_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct(
            "forecast_run_id"
        ).alias("distinct_run_count"),
        F.countDistinct(
            "_source_file_name"
        ).alias("source_file_count"),
        F.countDistinct(
            "_record_hash"
        ).alias("distinct_hash_count"),
        F.min("run_time_utc").alias(
            "min_run_time_utc"
        ),
        F.max("run_time_utc").alias(
            "max_run_time_utc"
        ),
        F.min(
            "forecast_valid_time_utc"
        ).alias("min_valid_time_utc"),
        F.max(
            "forecast_valid_time_utc"
        ).alias("max_valid_time_utc"),
        F.sum(
            (
                F.col("run_time_utc").isNull()
                |
                F.col(
                    "available_at_utc"
                ).isNull()
                |
                F.col(
                    "forecast_valid_time_utc"
                ).isNull()
            ).cast("int")
        ).alias("missing_timestamp_count"),
        F.sum(
            (
                F.col(
                    "lead_time_from_run_hours"
                ) < 0
            ).cast("int")
        ).alias("negative_lead_time_count"),
        F.sum(
            (
                F.col(
                    "lead_time_from_run_hours"
                ) >= FORECAST_HOURS
            ).cast("int")
        ).alias("invalid_lead_time_count"),
        F.sum(
            (
                F.col("available_at_utc")
                != (
                    F.col("run_time_utc")
                    + F.expr(
                        "INTERVAL 6 HOURS"
                    )
                )
            ).cast("int")
        ).alias(
            "availability_rule_violation_count"
        ),
        F.sum(
            (
                F.col("is_usable_for_replay")
                != (
                    F.col(
                        "forecast_valid_time_utc"
                    )
                    >= F.col(
                        "available_at_utc"
                    )
                )
            ).cast("int")
        ).alias(
            "usable_flag_mismatch_count"
        ),
        F.sum(
            (
                ~F.hour(
                    "run_time_utc"
                ).isin(0, 12)
            ).cast("int")
        ).alias("invalid_run_hour_count"),
        F.sum(
            F.col(
                "global_tilted_irradiance_instant_wm2"
            ).isNull().cast("int")
        ).alias("missing_gti_count")
    )
)

display(bronze_quality_df)

quality_result = (
    bronze_quality_df
    .first()
    .asDict()
)

test_errors = []

if (
    quality_result["row_count"]
    != EXPECTED_FORECAST_ROW_COUNT
):
    test_errors.append(
        "Niepoprawna liczba rekordów."
    )

if (
    quality_result["distinct_run_count"]
    != EXPECTED_RUN_COUNT
):
    test_errors.append(
        "Niepoprawna liczba runów."
    )

if (
    quality_result["source_file_count"]
    != EXPECTED_RUN_COUNT
):
    test_errors.append(
        "Niepoprawna liczba plików."
    )

if (
    quality_result["distinct_hash_count"]
    != EXPECTED_FORECAST_ROW_COUNT
):
    test_errors.append(
        "Hash rekordów nie jest unikalny."
    )

zero_expected_tests = {
    "wrong_rows_per_run_count":
        wrong_rows_per_run_count,
    "duplicate_business_key_count":
        duplicate_business_key_count,
    "missing_timestamp_count":
        quality_result[
            "missing_timestamp_count"
        ],
    "negative_lead_time_count":
        quality_result[
            "negative_lead_time_count"
        ],
    "invalid_lead_time_count":
        quality_result[
            "invalid_lead_time_count"
        ],
    "availability_rule_violation_count":
        quality_result[
            "availability_rule_violation_count"
        ],
    "usable_flag_mismatch_count":
        quality_result[
            "usable_flag_mismatch_count"
        ],
    "invalid_run_hour_count":
        quality_result[
            "invalid_run_hour_count"
        ],
}

for test_name, test_value in (
    zero_expected_tests.items()
):
    if test_value != 0:
        test_errors.append(
            f"{test_name} = {test_value}"
        )

if test_errors:
    raise RuntimeError(
        "Testy Bronze pogody niezaliczone: "
        + "; ".join(test_errors)
    )

print(
    "Wszystkie testy Bronze pogody "
    "zostały zaliczone."
)

(
    weather_forecast_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_WEATHER_TABLE)
)

saved_weather_bronze_df = spark.table(
    BRONZE_WEATHER_TABLE
)

saved_row_count = (
    saved_weather_bronze_df.count()
)

if (
    saved_row_count
    != EXPECTED_FORECAST_ROW_COUNT
):
    raise RuntimeError(
        "Po zapisie tabela zawiera "
        f"{saved_row_count} rekordów zamiast "
        f"{EXPECTED_FORECAST_ROW_COUNT}."
    )

print(
    f"Zapisano tabelę: "
    f"{BRONZE_WEATHER_TABLE}"
)

print(
    f"Liczba zapisanych rekordów: "
    f"{saved_row_count}"
)